# Data Viewer

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

## Acoustic Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as signal
from tritonoa.data.reader import read_hdf5, read_inventory
from tritonoa.data.time import TIME_PRECISION

from vineyard.config import get_path
from vineyard.plotting import plot_spectrogram

In [ ]:
# sensor = "3dvha"
# time_start = np.datetime64("2023-12-01T23:14:28.0", TIME_PRECISION)
# time_end = np.datetime64("2023-12-01T23:14:29.0", TIME_PRECISION)
# channels = 7

# sensor = "vla1"
# time_start = np.datetime64("2023-12-01T23:14:28.0", TIME_PRECISION)
# time_end = np.datetime64("2023-12-01T23:14:29.0", TIME_PRECISION)
# channels = 3

sensor = "vla2"
time_start = np.datetime64("2023-12-01T23:14:26.25", TIME_PRECISION)
time_end = np.datetime64("2023-12-01T23:14:27.25", TIME_PRECISION)
channels = 0

inventory = get_path(f"{sensor}_inventory")

ds = read_inventory(
    inventory,
    time_start=time_start,
    time_end=time_end,
    channels=channels,
).filter("bandpass", [15.0, 30.0])
print(ds.stats.sampling_rate)



nperseg = 1024
hop = 4
nfft = 2 ** 14
fs = ds.stats.sampling_rate
channels = np.arange(ds.num_channels)
window = signal.windows.hann(nperseg)
STFT = signal.ShortTimeFFT(window, hop, fs, mfft=nfft, scale_to="psd")
Sxx = STFT.spectrogram(ds.data[0])
f = STFT.f
t = STFT.t(ds.num_samples)
tvec = time_start + np.array([np.timedelta64(int(time * 1e6), TIME_PRECISION) for time in t])

In [ ]:
fig, axes = plt.subplots(nrows=2, figsize=(14, 6), sharex=True)
ax = axes[0]
ax.plot(ds.time_vector, ds.data[0], label="Original Signal")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")

ax = axes[1]
plot_spectrogram(f, tvec, Sxx, ax=ax)
# ax.set_ylim(15, 30)
plt.tight_layout()
plt.show()